# Gate G2: the MPI checks of the physics run (phase 2)

**Settings: every sample here runs with the frozen hard + bulk tune**: the hand tune of
`config/OO_MCGlauber_MUSIC_isr_rhictune.xml`, frozen at js-contrib `788ca1e` (2026-11-24). The
cell below reads the values from the XML itself and the per-sample options from the jobs' own
files, so the table shows what actually ran.
- **Hard tune** (RHIC p+p 200 GeV, `analysis/OO-RHIC` tune): α_s(M_Z) 0.152 for the hard process
  and the MPI, PDF `pSet 20` (iMATTER's NNPDF3.1 NNLO), Matter Q₀ 2 GeV, C_v (vir_factor) 0.25,
  MPI pTmin 5 GeV. Pythia lowers this to about 1.7 GeV at 200 GeV, giving about 0.6 MPI per hard event.
- **Bulk tune** (set `g_v045_n215`): 3dMCGlauber ylossParam4 nodes 1.6 / 2.15 / 2.15 / 2.15,
  variance 0.45, shadowing 0.45, BG 5.0, remnant energy-loss fraction 0.536,
  τ_form 0.424 fm/c, MUSIC + iSS freeze-out at 150 MeV.
- **What the samples vary, and nothing else:** MPI on (the tune) or `PartonLevel:MPI = off`, and
  for `pp_bulk_switch` the hard-pair string veto set to 0.

**What G2 decides** (tuning page, "Physics run plan", MPI subsection):
1. **Which MPI setting phase 3 runs with.** The rule is the setting that best reproduces STAR's
   p+p underlying event (UE) in the same chain as O+O.
2. **The MPI systematic on the selection share**, from the vacuum R_CP with MPI on − off.
3. **Whether the selection share is an upper estimate.** It is if no setting gives jet events a
   UE above minimum bias as STAR's does (the positive soft–hard correlation, the "pedestal").
4. **Whether to run the central-enriched pair** (`OO_PHYS_CENTRAL_BMAX`, from `phys_mb`).

The numbers come from `g2_check.py` (`results/g2_check.json`). The notebook only plots and
tabulates them. These are model results, not vetted physics results.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

R = json.loads(Path("results/g2_check.json").read_text())
S, UE, VAC = R["settings"], R["ue"], R["vac"]
print(f"XML: {S['xml']}\n  last commit: {S['xml_commit']}"
      + ("\n  !! the XML has local changes since that commit" if S["xml_modified_since"] else "  (unchanged since)"))
print("\nHard tune (from the XML):")
for k, v in S["hard"]["Pythia (PythiaIsrGun)"].items():
    print(f"  Pythia  {k} = {v}")
for k, v in S["hard"].items():
    if not isinstance(v, dict):
        print(f"  {k}: {v}")
print("\nBulk tune (3dMCGlauber, MUSIC; from the XML):")
for k, v in S["bulk"].items():
    print(f"  {k}: {v}")

In [ ]:
# what each sample ran with on top of the XML (read from one finished job of each), and how far it is
def opts(j):
    if not j:
        return "(no finished job)"
    extra = [l for l in j["pythia_lines"] if not any(l.startswith(k) for k in S["hard"]["Pythia (PythiaIsrGun)"])]
    return (f"MPI {'OFF' if j['mpi_off'] else 'on'}, string veto {j['hard_pair_string_veto']}, subtraction "
            f"{j['subtract_hard_momentum']}, vacuum jet {j['vacuum_jet']}; extra Pythia lines {extra or '-'}; "
            f"xml_set {[x.split('/')[-1] for x in j['xml_set']] or '-'}")
print("Part 1 samples (p+p, 9 windows 5-55 GeV each):")
for v, r in (UE or {}).get("variants", {}).items():
    print(f"  {v:16s} {r['label']:24s} {r['files']} jobs, events per window "
          + " ".join(str(int(x)) for x in r["events_per_window"]) + f"\n  {'':16s} {opts(r['job'])}")
print("\nPart 2 samples (O+O, jet in vacuum):")
for w, Rw in (VAC or {}).get("windows", {}).items():
    for arm in ("jets_vac", "jets_vac_mpioff"):
        if arm in Rw:
            print(f"  {arm}_{w:7s} {Rw[arm]['n']:6d} events; {opts(Rw[arm]['job'])}")
print(f"\nminimum bias (classes): phys_mb, {VAC['mb']['n']} events, <N_coll> {VAC['mb']['ncoll_mean']:.2f}, "
      f"MBD coincidence {VAC['mb']['eps_mbd_coincidence']:.3f} (sPHENIX 0.914)")

## Part 1: the p+p underlying event in the production chain

**Why.** Two MPI effects on the selection bias pull in opposite directions:
- **Toward a larger bias.** The MPI initiators' momentum is subtracted from the colliding nucleons
  together with the hard pair's, so less energy is left for the strings. Jet events then look less
  active and move to peripheral classes.
- **Against the bias.** MPI particles add activity to jet events. STAR's p+p UE in jet events lies
  well above the minimum-bias level (the pedestal), and the subtraction alone cannot produce that.

The p+p UE measured in the same chain as O+O tests the second effect directly. That chain has
strings, MUSIC + iSS, hard-momentum subtraction and the O+O string rule.

**Measurement.** STAR, PRD 101, 052004 (2020), HEPData ins1771348, with `ue_check.py`'s definitions:
- the leading anti-k_T R = 0.6 jet from charged and neutral particles (pT, E_T > 0.2 GeV,
  |η| < 1), with |η_jet| < 0.4;
- charged particles with pT > 0.2 GeV and |η| < 1 in the toward (|Δφ| < 60°), transverse (60–120°)
  and away (> 120°) regions;
- densities dN/(dη dφ).

**Model.**
- **Event weights:** each pTHat window is weighted with σ_k/N_k.
- **Incomplete low bins:** leading-jet bins below 11 GeV (grey) are incomplete, because there is
  no pTHat < 5 GeV. The table at the end of this part shows each bin's share from the lowest window.
- **Minimum-bias level:** from `pp_tau0424`, the p+p minimum-bias stand-in with the final XML,
  same acceptance. Its NSD dN/dη at |η| < 0.5 agrees with STAR's (printed below), so it stands in
  for the data's minimum-bias level.
- **"Hard tune alone":** the vacuum p+p chain without strings or hydro (`ue_check.py` run `ref_tune`).

**The deciding numbers:**
1. the level against STAR in the complete bins;
2. the falling trend (STAR goes from 0.66 to 0.49 between 11 and 45 GeV);
3. whether jet events sit above the minimum-bias level, as STAR's do.

In [ ]:
star = UE["star"]
E = np.asarray(star["edges"]); lo, hi = E[:-1], E[1:]; x = 0.5 * (lo + hi); xe = 0.5 * (hi - lo)
comp = lo >= UE["complete_from_gev"]
COL = {"pp_bulk": "C0", "pp_bulk_mpioff": "C3", "pp_bulk_switch": "C2"}
mbl = UE["mb_level"]
d = mbl.get("dndeta_nsd_eta05", {})
print(f"minimum-bias level ({mbl['sample']}, {mbl['events']} events): charged pT > 0.2, |eta| < 1: "
      f"{mbl['all_pt02']:.3f} (all), {mbl['nsd_pt02']:.3f} +- {mbl['nsd_pt02_err']:.3f} (NSD) per deta dphi; "
      f"pT > 0.5: {mbl['nsd_pt05']:.3f} (NSD)")
if d:
    print(f"  check: NSD dN/deta |eta| < 0.5 model {d['model']:.2f}, STAR {d['star']:.2f} +- {d['star_err']:.2f}")

def star_band(ax, key, scale=1.0, label="STAR (stat + sys)"):
    v = np.asarray(star[key]["value"]) * scale
    e = np.hypot(star[key]["stat"], star[key]["sys"]) * scale
    ax.fill_between(np.repeat(E, 2)[1:-1], np.repeat(v - e, 2), np.repeat(v + e, 2), color="0.75", alpha=0.6, lw=0, label=label)
    ax.plot(x, v, "ks", ms=4)

def shade(ax):
    ax.axvspan(E[0], UE["complete_from_gev"], color="0.92", zorder=-5)

def model(ax, key, ratio_to=None, ls="o", **kw):
    for i, (v, r) in enumerate(UE["variants"].items()):
        y = np.array([row[key] for row in r["rows"]]); ye = np.array([row.get(key + "_err", np.nan) for row in r["rows"]])
        if ratio_to is not None:
            y, ye = y / ratio_to, ye / ratio_to
        ax.errorbar(x + (i - 1) * 0.4, y, ye, xe * 0, fmt=ls, color=COL[v], ms=4, capsize=2, label=f"{v} ({r['label']})")

fig, (a, b) = plt.subplots(2, 1, figsize=(7.5, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
shade(a); star_band(a, "transverse"); model(a, "n_transverse")
if UE.get("hard_only"):
    a.plot(x, [r["n_transverse"] for r in UE["hard_only"]["rows"]], "k--", lw=1, label=UE["hard_only"]["label"])
a.axhspan(mbl["nsd_pt02"] - mbl["nsd_pt02_err"], mbl["nsd_pt02"] + mbl["nsd_pt02_err"], color="C1", alpha=0.4)
a.axhline(mbl["nsd_pt02"], color="C1", lw=1, label="minimum-bias level (p+p NSD, same chain)")
a.set_ylabel(r"transverse $dN_{ch}/d\eta\,d\phi$  ($p_T > 0.2$ GeV)"); a.set_ylim(0, 1.1); a.legend(fontsize=7.5, loc="upper right")
a.set_title("p+p 200 GeV underlying event, frozen hard + bulk tune (788ca1e)", fontsize=10)
shade(b)
sv = np.asarray(star["transverse"]["value"]); se = np.hypot(star["transverse"]["stat"], star["transverse"]["sys"]) / sv
b.fill_between(np.repeat(E, 2)[1:-1], np.repeat(1 - se, 2), np.repeat(1 + se, 2), color="0.75", alpha=0.6, lw=0)
model(b, "n_transverse", ratio_to=sv); b.axhline(1, color="k", lw=0.8)
b.set_ylabel("model / STAR"); b.set_xlabel("leading-jet $p_T$ [GeV]"); b.set_ylim(0.2, 1.6)
plt.tight_layout(); plt.show()

**Figure 1.** Transverse charged density vs leading-jet pT. The grey area marks bins that are
incomplete in the model. The orange line is the model's minimum-bias level, which matches the
data's minimum-bias dN/dη.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7.5))
for a_, key, sk, ttl in ((ax[0, 0], "n_toward", "toward", "toward"), (ax[0, 1], "n_away", "away", "away"),
                         (ax[1, 0], "n_trans05", "transverse05", r"transverse, $p_T > 0.5$ GeV")):
    shade(a_); star_band(a_, sk); model(a_, key); a_.set_title(ttl, fontsize=10)
    a_.set_ylabel(r"$dN_{ch}/d\eta\,d\phi$"); a_.set_xlabel("leading-jet $p_T$ [GeV]")
ax[1, 0].axhline(mbl["nsd_pt05"], color="C1", lw=1, label="minimum-bias level")
a_ = ax[1, 1]; shade(a_)
for i, (v, r) in enumerate(UE["variants"].items()):
    a_.plot(x, [row["n_transmax"] for row in r["rows"]], "^-", color=COL[v], ms=4, label=f"{v} TransMax")
    a_.plot(x, [row["n_transmin"] for row in r["rows"]], "v--", color=COL[v], ms=4, label=f"{v} TransMin")
a_.axhline(mbl["nsd_pt02"], color="C1", lw=1); a_.set_title("TransMax / TransMin (one side each; no STAR points loaded)", fontsize=10)
a_.set_xlabel("leading-jet $p_T$ [GeV]")
for a_ in ax.flat:
    a_.legend(fontsize=6.5)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
a_ = ax[0]; shade(a_)
if "meanpt_transverse" in star:
    star_band(a_, "meanpt_transverse")
for v, r in UE["variants"].items():
    a_.plot(x, [row["meanpt_transverse"] for row in r["rows"]], "o-", color=COL[v], ms=4, label=v)
a_.set_ylabel(r"transverse $\langle p_T\rangle$ [GeV]  ($p_T > 0.2$)"); a_.set_xlabel("leading-jet $p_T$ [GeV]"); a_.legend(fontsize=7)
a_ = ax[1]
for v, r in UE["variants"].items():
    a_.plot(x, [row["has_bulk"] for row in r["rows"]], "o-", color=COL[v], ms=4, label=v)
a_.set_ylabel("fraction of jet events with a medium"); a_.set_xlabel("leading-jet $p_T$ [GeV]"); a_.set_ylim(0, 1.05); a_.legend(fontsize=7)
plt.tight_layout(); plt.show()

**Figure 2.** The other regions:
- the toward and away regions, which hold the jet and the recoil and test the hard side;
- the transverse density for pT > 0.5 GeV;
- TransMax and TransMin (TransMin is the cleanest soft-UE measure);
- the transverse ⟨pT⟩;
- the fraction of jet events that had a medium at all. With the veto on (`pp_bulk`,
  `pp_bulk_mpioff`), a p+p jet event gets a string only if a nucleon–nucleon pair is left after the
  hard pair. `pp_bulk_switch` puts a string in every jet event.

In [ ]:
print(f"complete bins: leading jet >= {UE['complete_from_gev']:.0f} GeV ({int(comp.sum())} bins)")
print(f"{'sample':16s} {'<model/STAR>':>13s} {'chi2/bins':>10s}  {'trend first/last':>16s} (STAR {UE['variants'][next(iter(UE['variants']))]['summary']['star_trend_first_over_last']:.2f})   UE / minimum bias per complete bin")
for v, r in UE["variants"].items():
    s = r["summary"]
    print(f"{v:16s} {s['mean_ratio_to_star']:13.2f} {s['chi2_complete_bins']:6.1f}/{s['n_bins']:<3d}  {s['trend_first_over_last']:16.2f}           "
          + " ".join(f"{q:.2f}" for q in s["over_mb_nsd"]))
print(f"{'STAR':16s} {'':13s} {'':10s}  {'':16s}           " + " ".join(f"{q:.2f}" for q in UE["star_over_mb_nsd"]))
if UE.get("hard_only"):
    h = np.array([r["n_transverse"] for r in UE["hard_only"]["rows"]])
    print(f"{'hard tune alone':16s} {np.mean(h[comp] / sv[comp]):13.2f}")
print("\nshare of each leading-jet bin from the lowest window (5-7 GeV; large = the bin also needs pTHat < 5):")
for v, r in UE["variants"].items():
    print(f"  {v:16s} " + " ".join(f"{100 * q:4.0f}%" for q in r["low_window_share"]))

## Part 2: the O+O vacuum bias with MPI on and off

**Why.** The p+p UE says how well each setting describes the soft–hard correlation. What O+O needs
is the size of MPI's effect on the quantity being decomposed: the **vacuum R_CP**, the selection
bias alone.

**Model.**
- **Samples:** `jets_vac_w18_23` and `_w30_40` with MPI on (the start of the phase-3 vacuum
  samples), and `jets_vac_mpioff_*`, which reuse their seeds.
- **Classes:** percentiles of `phys_mb` (50k minimum-bias events, same XML). STAR's classes use
  the EPD (2.1 < |η| < 5.1) and sPHENIX's use the MBD (3.51 < |η| < 4.61, both arms).
- **Observables:** jet events weighted by N_coll; R_AA^vac = (f_jet,c / f_MB,c) ⟨N_coll⟩_MB /
  ⟨N_coll⟩_MB,c per class, and R_CP from it (`oo_bias.class_analysis`, the same random tie-break
  for both arms).
- **Errors on on − off:** the arms start from the same seeds but do not share geometries event by
  event, because MPI off changes the random-number stream (`same_geometry_fraction` below). So
  on − off carries independent errors.
- **Mid-rapidity estimator:** |η| < 1 contains the jet itself (autocorrelation), so it is only
  tabulated.

**Reference (R4b, MAP bulk set, other windows):** vacuum R_CP 0.876 (MBD) and 0.951 (EPD).

In [ ]:
SC, PC = VAC["star_classes"], VAC["sph_classes"]
ARMC = {"jets_vac": "C0", "jets_vac_mpioff": "C3"}
ARML = {"jets_vac": "MPI on (tune)", "jets_vac_mpioff": "MPI off"}
wins = list(VAC["windows"])
fig, ax = plt.subplots(2, len(wins), figsize=(5.5 * len(wins), 7), squeeze=False)
for j, w in enumerate(wins):
    Rw = VAC["windows"][w]
    for i, (est, cls, ttl) in enumerate((("epd", SC, "STAR classes, EPD"), ("mbd", PC, "sPHENIX classes, MBD"))):
        a_ = ax[i, j]
        xc = np.arange(len(cls))
        for k, arm in enumerate(("jets_vac", "jets_vac_mpioff")):
            if arm not in Rw:
                continue
            c = Rw[arm]["est"][est]
            a_.errorbar(xc + (k - 0.5) * 0.15, c["raa_vac"], c["raa_err"], fmt="o", color=ARMC[arm], capsize=2,
                        label=f"{ARML[arm]} ({Rw[arm]['n']} ev)")
        a_.axhline(1, color="k", lw=0.8)
        a_.set_xticks(xc); a_.set_xticklabels([f"{a}-{b}%" for a, b in cls], fontsize=8)
        a_.set_title(f"pTHat {w[1:].replace('_', '-')} GeV, {ttl}", fontsize=10); a_.set_ylabel(r"$R_{AA}^{vac}$")
        a_.set_ylim(0.5, 1.5); a_.legend(fontsize=7.5)
plt.suptitle("O+O 200 GeV, jets in vacuum: selection bias per class, frozen hard + bulk tune (788ca1e)", fontsize=10)
plt.tight_layout(); plt.show()

**Figure 3.** Vacuum R_AA per centrality class, with MPI on and off. Without a bias every class
would be at 1. A falling pattern from peripheral to central classes is the selection bias (central
classes below 1).

In [ ]:
keys = [("epd", "0-10/60-80"), ("epd", "0-10/40-60"), ("mbd", "0-20/60-80"), ("mbd", "0-20/40-60")]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for j, w in enumerate(wins):
    Rw = VAC["windows"][w]
    for k, arm in enumerate(("jets_vac", "jets_vac_mpioff")):
        if arm not in Rw:
            continue
        v = [Rw[arm]["est"][e]["rcp"][c] for e, c in keys]
        ax[0].errorbar(np.arange(len(keys)) + 0.1 * (2 * j + k) - 0.15, [a for a, _ in v], [b for _, b in v], fmt="os"[j],
                       color=ARMC[arm], mfc="none" if j else ARMC[arm], capsize=2, label=f"{w} {ARML[arm]}")
    if "rcp_on_minus_off" in Rw:
        dd = [Rw["rcp_on_minus_off"][f"{e} {c}"] for e, c in keys]
        ax[1].errorbar(np.arange(len(keys)) + 0.15 * j - 0.075, [a for a, _ in dd], [b for _, b in dd], fmt="os"[j], color="k",
                       mfc="none" if j else "k", capsize=2, label=w)
for a_, yl in ((ax[0], r"vacuum $R_{CP}$"), (ax[1], r"$R_{CP}$(MPI on) $-$ $R_{CP}$(MPI off)")):
    a_.set_xticks(range(len(keys))); a_.set_xticklabels([f"{e.upper()} {c}" for e, c in keys], fontsize=8)
    a_.set_ylabel(yl); a_.legend(fontsize=7)
ax[0].axhline(1, color="k", lw=0.8); ax[1].axhline(0, color="k", lw=0.8)
plt.tight_layout(); plt.show()

print(f"{'window':7s} {'estimator / R_CP':22s} {'MPI on':>14s} {'MPI off':>14s} {'on - off':>15s}")
for w in wins:
    Rw = VAC["windows"][w]
    if "rcp_on_minus_off" not in Rw:
        continue
    for e in ("epd", "mbd", "mid10"):
        for c in Rw["jets_vac"]["est"][e]["rcp"]:
            a, ea = Rw["jets_vac"]["est"][e]["rcp"][c]; b, eb = Rw["jets_vac_mpioff"]["est"][e]["rcp"][c]
            dv, de = Rw["rcp_on_minus_off"][f"{e} {c}"]
            print(f"{w:7s} {e + ' ' + c:22s} {a:7.3f}+-{ea:.3f} {b:7.3f}+-{eb:.3f} {dv:+8.3f}+-{de:.3f}" + ("   (contains the jet)" if e == "mid10" else ""))
    print(f"{w:7s} MBD coincidence of jet events (N_coll-weighted): on {Rw['jets_vac']['eps_mbd_coincidence_weighted']:.3f}, "
          f"off {Rw['jets_vac_mpioff']['eps_mbd_coincidence_weighted']:.3f}; same geometry event by event: "
          f"{100 * Rw['same_geometry_fraction']:.1f} % of {Rw['paired_events']}")

**Figure 4.** Vacuum R_CP with MPI on and off (left), and their difference (right), for STAR's EPD
classes and sPHENIX's MBD classes. The difference is the MPI systematic on the selection share.

In [ ]:
ed = np.asarray(VAC["epd_vs_ncoll"]["edges"]); xc = 0.5 * (ed[:-1] + ed[1:])
fig, ax = plt.subplots(1, len(wins), figsize=(5.5 * len(wins), 3.8), squeeze=False)
for j, w in enumerate(wins):
    a_ = ax[0, j]; Rw = VAC["windows"][w]
    mbv = np.asarray(VAC["epd_vs_ncoll"]["mb"])
    a_.axhline(1, color="C1", lw=1, label="minimum bias (phys_mb)")
    for arm in ("jets_vac", "jets_vac_mpioff"):
        if arm in Rw:
            a_.errorbar(xc, np.asarray(Rw[arm]["epd_vs_ncoll"]) / mbv, np.asarray(Rw[arm]["epd_vs_ncoll_err"]) / mbv,
                        fmt="o-", color=ARMC[arm], ms=4, capsize=2, label=ARML[arm])
    a_.set_xscale("log"); a_.set_xlabel(r"$N_{coll}$"); a_.set_ylabel(r"$\langle N_{EPD}\rangle$ jet events / minimum bias")
    a_.set_title(f"pTHat {w[1:].replace('_', '-')} GeV", fontsize=10); a_.legend(fontsize=7.5)
plt.tight_layout(); plt.show()

**Figure 5. The mechanism:** EPD multiplicity of jet events relative to minimum bias at the same
N_coll.
- **Below 1:** the hard-momentum subtraction (the energy taken by the hard pair and the MPI
  initiators) makes jet events less active forward. This moves them to peripheral classes and is
  the selection bias.
- **MPI on vs off:** the gap between the two curves is MPI's net effect in the forward
  estimator: the extra subtraction against the extra particles.

## Summary for the gate

The cell below applies the page's decision rule to the numbers mechanically. The reading
underneath is written at G2 with the complete phase-2 statistics.

**Decision rule** (tuning page):
- Use the MPI setting that best reproduces STAR's p+p UE in the same chain.
- Quote the vacuum R_CP difference between MPI on and off as the systematic on the selection share.
- If no setting reaches STAR's UE, part of the positive correlation is missing in every setting,
  and the model's selection share is an upper estimate.

In [ ]:
var = UE["variants"]
best = min(var, key=lambda v: var[v]["summary"]["chi2_complete_bins"] / var[v]["summary"]["n_bins"])
print(f"1. best p+p UE (chi2 over the {var[best]['summary']['n_bins']} complete transverse bins): {best} ({var[best]['label']}); "
      + ", ".join(f"{v} {r['summary']['chi2_complete_bins']:.1f}" for v, r in var.items()))
for v, r in var.items():
    s = r["summary"]; above = np.mean(s["over_mb_nsd"])
    print(f"   {v:16s} level {s['mean_ratio_to_star']:.2f} x STAR, trend {s['trend_first_over_last']:.2f} (STAR {s['star_trend_first_over_last']:.2f}), "
          f"UE / minimum bias {above:.2f} (STAR {np.mean(UE['star_over_mb_nsd']):.2f})")
print("2. MPI systematic on the vacuum R_CP (on - off):")
for w in wins:
    Rw = VAC["windows"][w]
    if "rcp_on_minus_off" in Rw:
        print(f"   {w}: " + ", ".join(f"{k} {a:+.3f} +- {b:.3f}" for k, (a, b) in Rw["rcp_on_minus_off"].items() if not k.startswith("mid10")))
ok = [v for v, r in var.items() if abs(r["summary"]["mean_ratio_to_star"] - 1) < 0.15
      and abs(r["summary"]["trend_first_over_last"] - r["summary"]["star_trend_first_over_last"]) < 0.15]
print("3. settings within 15 % of STAR's level and trend: " + (", ".join(ok) if ok else "none -> selection share = upper estimate"))
print("4. central pair: b_max 5.5 fm contains every 0-20 % event (EPD, MBD, |eta| < 1); enrichment 1.28x per generated event")

### Reading (2026-11-24, complete phase 2)

Statistics: 3000 p+p events per window × 9 windows × 3 settings, and 5000 O+O events per arm and
window. Leading-jet bins from 11 GeV. All samples ran with the frozen hard + bulk tune; only the
MPI switch and the string veto differ.

**1. The p+p UE: MPI (as tuned) supplies the pedestal, most of its level but not its trend.**
- **MPI on (the tune)** is the best of the three settings (χ² 26 / 5 bins):
  - Transverse density 0.70 → 0.91 × STAR from 11–15 to 35–45 GeV.
  - Jet events at 1.5 × the minimum-bias level, against STAR's 2.15 → 1.60.
  - So the positive soft–hard correlation is there, but too weak at low leading-jet pT. The
    model's UE is flat (first/last 1.03), while STAR's falls (1.34).
- **MPI off:** the UE sits at the minimum-bias level (1.0 ×, 0.57 × STAR), the same as the hard
  tune alone. Strings and hydro give no pedestal by themselves, as Track U expected. Only about
  35 % of p+p jet events have a medium with the veto on.
- **String veto 0:** a string in every jet event overshoots (1.23 × STAR, 2.2 × minimum bias),
  and it is still flat.
- **All three are too soft:** the transverse ⟨pT⟩ is about 0.8 × STAR in every setting, and the
  pT > 0.5 GeV density is 0.56–0.72 × STAR with MPI on. Most of the missing UE is above 0.5 GeV.
- **Toward and away regions:** with MPI on they are 0.90–0.98 × STAR, within STAR's systematics
  at higher leading-jet pT. The hard side is fine.

**2. The MPI systematic on the selection bias is small: R_CP(on) − R_CP(off) is consistent with
zero.**
- **Vacuum R_CP with MPI on** (the selection bias alone):
  - MBD 0–20/60–80: 0.877 ± 0.046 (18–23 GeV) and 0.885 ± 0.048 (30–40 GeV).
  - EPD 0–10/60–80: 0.964 ± 0.058 and 0.950 ± 0.058.
  - These are the R4b values (0.876 MBD, 0.951 EPD, MAP bulk set), so the bulk retune did not
    change the size of the bias.
- **On − off, both windows combined:**
  - MBD 0–20/60–80: −0.020 ± 0.048. MBD 0–20/40–60: +0.005 ± 0.047.
  - EPD 0–10/60–80: +0.046 ± 0.057. EPD 0–10/40–60: +0.012 ± 0.058.
  - No window or estimator differs by more than 1.6σ.
  - The systematic to quote is ±0.05 (MBD) and ±0.06 (EPD), the statistical limit of this test.
- **Why MPI matters little here even though it matters in p+p** (Figure 5):
  - MPI off makes jet events at N_coll 1–3 5–8 % (18–23 GeV) to 11–21 % (30–40 GeV) less active
    in the EPD than minimum bias. MPI on brings them close to it.
  - Above N_coll ≈ 5 both settings agree within a few %.
  - The N_coll-weighted jet events sit mostly at large N_coll, so R_CP barely moves.
- **MBD coincidence of jet events:** 0.97 in both settings.

- **The sign of MPI's effect differs from the proxy.** In this full chain, MPI makes low-N_coll
  jet events *more* active in the EPD than MPI off. The MPI hadrons outweigh the extra
  subtraction, which is the opposite of the proxy statement "MPI only lower the activity" on the
  tuning page.

**3. The MPI choice (the user's decision at the gate). There are two readings of the rule:**
- **(a) Keep MPI on.** It is the best of the settings tested in this chain, which is the original
  rule. `oo_bias.py` stays unchanged, and the phase-2 "on" arms fold into phase 3.
- **(b) Soft MPI.** This follows the page's "G2 decision input", which is based on the GB10
  pre-check A: MPI on stays clearly below STAR at 6–15 GeV (0.68–0.70 ×), the expected case.
  - The setting: MPI α_s 0.130, `pTmin` 0.2, `ecmRef` 200, `pT0Ref` ≈ 1.42–1.45.
  - First a p+p hydro check, about 27k events and 1 h on the RTX box.
  - Then the soft lines go into `oo_bias.py`, and phase 3 needs about 37k extra events. The
    phase-2 "on" arms become the systematic sample.
  - It brings the UE level to STAR's, but not the trend.
- **For the selection share the choice matters little.** MPI on − off is ±0.05 (MBD) and
  ±0.06 (EPD), and soft vs on is ≈ 0.013 in the GB10 proxy.
- **Either way, the selection share is quoted as an upper estimate.** No setting reaches STAR's UE
  in level and trend: the pedestal is about 30 % short at low leading-jet pT. The veto-0 setting
  overshoots STAR in p+p, so in O+O it could bound the selection share from the other side; it
  was not run in O+O.

**4. Open for the user's decision at the gate:**
- **The O+O string-switch bracket.** In p+p, veto 0 brackets STAR from above, and MPI on from
  below. An O+O `jets_vac_switch` in the same two windows (2 × 5k, about 0.8 h) would give the
  selection share on the other side of the data.
- **The central pair.** b_max 5.5 fm keeps every 0–20 % event but enriches only 1.28 ×. Run it as
  planned, or put the same events into the phase-3 windows 7–18 GeV.

Model results, not vetted physics results.